In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
%cd /kaggle/working
!rm -rf /tmp/repo && git clone https://github.com/amanchaurasia512/Self_optimized_Agent_Ai_chunking.git /tmp/repo && git -C /tmp/repo checkout -q 18b62a1 && git -C /tmp/repo log --oneline -1
!pip install jsonschema langchain-text-splitters==1.1.3 langchain-experimental==0.4.2 langchain-community==0.4.2 sentence-transformers==6.2.0 transformers==5.22.1 stanza==1.15.0 2>&1 | tail -n 15
%cd /tmp/repo

In [ ]:
import shutil
from importlib.metadata import version
from pathlib import Path

import torch

print("GPU:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
print({name: version(name) for name in ["torch", "transformers", "sentence-transformers", "langchain-experimental", "langchain-community"]})
for parser in ["docling", "pdfplumber"]:
    source = next(path.parent for path in Path("/kaggle/input").rglob("indian_d1.json") if path.parent.name == parser)
    shutil.copytree(source, f"/tmp/repo/data/extracted/{parser}", dirs_exist_ok=True)
    print(parser, len(list(Path(f"/tmp/repo/data/extracted/{parser}").glob("*.json"))), "files from", source)

In [ ]:
%cd /tmp/repo
!python -m scripts.p2_chunk --chunker semantic --machine kaggle-gpu-t4 --run-source KN01
!mkdir -p /kaggle/working/chunks && cp -r data/chunks/docling /kaggle/working/chunks/ && cp -r data/chunks/pdfplumber /kaggle/working/chunks/ && cp runs/P2-KN01-*.json /kaggle/working/
!ls -R /kaggle/working/chunks | head -20